In [1]:
import numpy as np
from typing import Any
import os
import hail as hl
import pyspark.sql.functions as f
import pandas as pd

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.l2g_prediction import L2GPrediction
hail_dir = os.path.dirname(hl.__file__)
session = Session(hail_home=hail_dir, start_hail=True, extended_spark_conf={"spark.driver.memory": "12g","spark.kryoserializer.buffer.max": "500m","spark.driver.maxResultSize":"2g"})
hl.init(sc=session.spark.sparkContext, log="/dev/null")

Loading BokehJS ...

24/04/10 16:11:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


24/04/10 16:11:42 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.3.4
SparkUI available at http://mib118093s.internal.sanger.ac.uk:4041
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.127-bb535cd096c5
LOGGING: writing to /dev/null


In [2]:
path_to_cs="gs://genetics_etl_python_playground/releases/24.03/credible_set/gwas_catalog_PICSed_curated_associations/"
cred_sets=StudyLocus.from_parquet(session=session,path=path_to_cs)

In [3]:
cred_sets.df.count()

552202

In [4]:
df=cred_sets.df.filter(f.col("studyId")=="GCST90105038")
df.count()

4829

In [5]:
df.show()

+--------------------+----------------+----------+---------+------+------------+-----------+--------------+--------------+-------------------------------+-------------+-------------------+--------------------+-----------------+----------------+------------------+----------+--------------------+--------------------+
|        studyLocusId|       variantId|chromosome| position|region|     studyId|       beta|pValueMantissa|pValueExponent|effectAlleleFrequencyFromSource|standardError|subStudyDescription|     qualityControls|finemappingMethod|credibleSetIndex|credibleSetlog10BF|sampleSize|               ldSet|               locus|
+--------------------+----------------+----------+---------+------+------------+-----------+--------------+--------------+-------------------------------+-------------+-------------------+--------------------+-----------------+----------------+------------------+----------+--------------------+--------------------+
| 6757354732808326276|            null|         3

In [6]:
path_to_l2g="gs://genetics_etl_python_playground/releases/24.03/locus_to_gene_predictions"
l2g=session.read_parquet(path_to_l2g,schema=L2GPrediction.get_schema())

In [7]:
l2g.show()

+--------------------+---------------+--------------------+
|        studyLocusId|         geneId|               score|
+--------------------+---------------+--------------------+
|-6895902971692467050|ENSG00000134824|  0.7080287337303162|
|-6895902971692467050|ENSG00000134824|  0.7080287337303162|
| 8497207144013884889|ENSG00000177311| 0.14295943081378937|
| 6463695232794599647|ENSG00000081154|0.003860333468765...|
| 6463695232794599647|ENSG00000081154|0.003860333468765...|
|-9129898941855357263|ENSG00000250644| 0.04913777485489845|
| 5671670686287878413|ENSG00000182271| 0.14377844333648682|
|-8628709953537504477|ENSG00000171862|  0.8795352578163147|
| -775069249032853922|ENSG00000204351|0.003774740733206272|
|-7363908191509227866|ENSG00000163945|  0.2766507565975189|
|-7363908191509227866|ENSG00000163945|  0.2766507565975189|
| 2484433207148505478|ENSG00000204709|  0.6865723729133606|
| 8466124716085818415|ENSG00000118655|  0.5453958511352539|
| 8466124716085818415|ENSG00000118655|  

In [8]:
l2g.count()

10561869

In [9]:
df_merge=df.join(l2g,"studyLocusId",how="inner")

In [10]:
df_merge.count()

38336

In [11]:
df_merge.show()

+--------------------+--------------+----------+--------+------+------------+---------+--------------+--------------+-------------------------------+-------------+-------------------+---------------+-----------------+----------------+------------------+----------+--------------------+--------------------+---------------+--------------------+
|        studyLocusId|     variantId|chromosome|position|region|     studyId|     beta|pValueMantissa|pValueExponent|effectAlleleFrequencyFromSource|standardError|subStudyDescription|qualityControls|finemappingMethod|credibleSetIndex|credibleSetlog10BF|sampleSize|               ldSet|               locus|         geneId|               score|
+--------------------+--------------+----------+--------+------+------------+---------+--------------+--------------+-------------------------------+-------------+-------------------+---------------+-----------------+----------------+------------------+----------+--------------------+--------------------+------

In [12]:
df_final=df_merge.filter(f.col("score")>=0.5)

In [13]:
df_final.count()

2776

In [15]:
df_grouped=df_final.groupBy("geneId").agg(f.count("studyLocusId").alias("num_loci"))
df_grouped.count()

2178

In [16]:
out=df_final.toPandas()

In [17]:
out

,studyLocusId,variantId,chromosome,position,region,studyId,beta,pValueMantissa,pValueExponent,effectAlleleFrequencyFromSource,...,subStudyDescription,qualityControls,finemappingMethod,credibleSetIndex,credibleSetlog10BF,sampleSize,ldSet,locus,geneId,score
0,-9221662644183536443,3_50557710_C_T,3,50557710,None,GCST90105038,0.013433,6.0,-15,NaN,...,None,[],pics,NaN,NaN,NaN,"[(3_50515435_G_A, 0.5785817115879058), (3_5063...","[(True, True, None, 0.828191509820447, 3_50557...",ENSG00000088538,0.529466
1,-9103997315260646097,16_28263483_A_G,16,28263483,None,GCST90105038,0.006507,7.0,-9,NaN,...,None,[],pics,NaN,NaN,NaN,"[(16_28282824_C_CTG, 0.0), (16_28263453_C_G, 0...","[(True, True, None, 0.5733761938805333, 16_282...",ENSG00000103522,0.686572
2,-8963100601246627350,20_8422903_T_C,20,8422903,None,GCST90105038,-0.010367,9.0,-16,NaN,...,None,[],pics,NaN,NaN,NaN,"[(20_8410085_T_A, 0.6902316941250257), (20_841...","[(True, True, None, 0.585267479656945, 20_8422...",ENSG00000182621,0.997903
3,-8797854827954997795,5_107714762_A_G,5,107714762,None,GCST90105038,0.006387,1.0,-9,NaN,...,None,[],pics,NaN,NaN,NaN,"[(5_107699045_C_T, 0.0), (5_107714332_A_G, 0.5...","[(True, True, None, 0.7364176334175184, 5_1077...",ENSG00000184349,0.507169
4,-8735406506619596200,2_232901387_G_A,2,232901387,None,GCST90105038,-0.012850,6.0,-12,NaN,...,None,[],pics,NaN,NaN,NaN,"[(2_232873607_T_C, 0.0), (2_232891160_G_A, 0.0...","[(True, True, None, 0.6159997295958188, 2_2329...",ENSG00000066248,0.888442
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2771,9026318706888160174,16_83418980_G_A,16,83418980,None,GCST90105038,-0.006286,4.0,-9,NaN,...,None,[],pics,NaN,NaN,NaN,"[(16_83418980_G_A, 1.0000000000000004), (16_83...","[(True, True, None, 0.7573110933917525, 16_834...",ENSG00000140945,0.755756
2772,9046550940660490420,3_71534232_G_A,3,71534232,None,GCST90105038,0.016546,4.0,-55,NaN,...,None,[],pics,NaN,NaN,NaN,"[(3_71478634_T_C, 0.7524763952501705), (3_7153...","[(True, True, None, 0.24103351940058207, 3_715...",ENSG00000114861,0.975454
2773,9084262932355854734,22_46806794_C_T,22,46806794,None,GCST90105038,0.008108,6.0,-12,NaN,...,None,[],pics,NaN,NaN,NaN,"[(22_46780114_G_C, 0.6503764859366227), (22_46...","[(True, True, None, 0.10497953366795715, 22_46...",ENSG00000054611,0.953972
2774,9177158647412534001,7_105606374_C_T,7,105606374,None,GCST90105038,0.006560,4.0,-10,NaN,...,None,[],pics,NaN,NaN,NaN,"[(7_105608673_A_C, 0.507091488445125), (7_1056...","[(True, True, None, 0.9986761228718036, 7_1056...",ENSG00000146776,0.561753


In [19]:
out.to_csv("~/test.tsv",sep="\t",index=False)